In [1]:
"""
MMLU Dev-Set Evaluation: Baseline vs Finetuned OLMo-2-7B
Runs both models sequentially and saves results to Drive.
Uses dev split (~285 examples/language) for fast evaluation (~10 mins total).

SETUP:
  - Run on A100 for best speed
  - Mount Google Drive before running
  - Adjust CKPT_PATH to point to your checkpoint folder
"""

# ── 1. Install ─────────────────────────────────────────────────────────────────
import subprocess, os, shutil, torch

!git clone https://github.com/EleutherAI/lm-evaluation-harness.git
%cd lm-evaluation-harness
!pip install -e ".[vllm]" -q

# ── 2. Swap test → dev in global_mmlu yaml files ───────────────────────────────
# Do this BEFORE importing lm_eval so the task registry picks up the change
print("Patching global_mmlu tasks to use dev split...")
!find /content/lm-evaluation-harness/lm_eval/tasks/global_mmlu -type f \
  -exec sed -i 's/test_split: test/test_split: dev/g' {} +

# Verify
!grep -r "test_split" /content/lm-evaluation-harness/lm_eval/tasks/global_mmlu/ | head -5

# Verify
result = subprocess.run(
    ["grep", "-r", "test_split", "/content/lm-evaluation-harness/lm_eval/tasks/global_mmlu/"],
    capture_output=True, text=True
)
print("Confirming dev split patch:")
for line in result.stdout.splitlines()[:3]:
    print(" ", line)

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 62862, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 62862 (delta 0), reused 0 (delta 0), pack-reused 62859 (from 2)
Receiving objects: 100% (62862/62862), 34.22 MiB | 9.55 MiB/s, done.
Resolving deltas: 100% (43425/43425), done.
/content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os, shutil, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Config
BASE_MODEL  = "allenai/OLMo-2-1124-7B"
CKPT_PATH   = "/content/drive/MyDrive/UCL/SNLP/olmo2-translation-lora-checkpoints/checkpoint-400"
MERGED_PATH = "/content/olmo2-lora-merged"
RESULTS_FT  = "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_finetuned_olmo2"

# ── 4. Merge LoRA adapter into base model ─────────────────────────────────────
if not os.path.exists(MERGED_PATH):
    print(f"\nMerging LoRA adapter from {CKPT_PATH}...")

    LOCAL_CKPT = "/content/lora-checkpoint"
    if os.path.exists(LOCAL_CKPT):
        shutil.rmtree(LOCAL_CKPT)
    shutil.copytree(CKPT_PATH, LOCAL_CKPT)

    print("Loading base model for merge...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    print("Applying LoRA adapter...")
    peft_model = PeftModel.from_pretrained(base, LOCAL_CKPT)
    merged     = peft_model.merge_and_unload()

    print(f"Saving merged model to {MERGED_PATH}...")
    merged.save_pretrained(MERGED_PATH)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(MERGED_PATH)

    del base, peft_model, merged
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print("Merge complete. GPU memory freed.")
else:
    print(f"Merged model already exists at {MERGED_PATH}, skipping merge.")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Merge complete. GPU memory freed.


In [4]:
import os

# Write the eval command to a shell script
# Notice: pretrained is now {MERGED_PATH} and output_path is {RESULTS_FT}
script = f"""#!/bin/bash
export VLLM_USE_V1=0
lm_eval --model vllm \\
  --model_args "pretrained={MERGED_PATH},dtype=auto,gpu_memory_utilization=0.75" \\
  --tasks global_mmlu_full_en global_mmlu_full_es global_mmlu_full_fr \\
          global_mmlu_full_de global_mmlu_full_id global_mmlu_full_pt \\
          global_mmlu_full_ru global_mmlu_full_zh global_mmlu_full_ja \\
          global_mmlu_full_ar global_mmlu_full_sw global_mmlu_full_bn \\
  --batch_size auto \\
  --output_path {RESULTS_FT} \\
  --include_path /content/lm-evaluation-harness/lm_eval/tasks
"""

with open("/content/run_eval_ft.sh", "w") as f:
    f.write(script)

os.system("chmod +x /content/run_eval_ft.sh")

print("Starting evaluation of the merged fine-tuned model...")

# Run in subprocess in the background
os.system("bash /content/run_eval_ft.sh > /content/eval_ft_log.txt 2>&1 &")

Starting evaluation of the merged fine-tuned model...


0

In [4]:
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer

# base = AutoModelForCausalLM.from_pretrained(
#         BASE_MODEL,
#         torch_dtype=torch.bfloat16,
#         device_map="auto",
#     )

# import torch, gc

# # Just run baseline eval directly — vllm handles model loading internally
# run_lm_eval(
#     model_path=BASE_MODEL,
#     output_path=RESULTS_BASE,
#     label="Baseline OLMo-2-7B",
# )

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model-00001-of-00006.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]


Evaluating: Baseline OLMo-2-7B
  Model:  allenai/OLMo-2-1124-7B
  Output: /content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_baseline_olmo2



CalledProcessError: Command '['lm_eval', '--model', 'vllm', '--model_args', 'pretrained=allenai/OLMo-2-1124-7B,dtype=auto,gpu_memory_utilization=0.7', '--tasks', 'global_mmlu_full_en', 'global_mmlu_full_es', 'global_mmlu_full_fr', 'global_mmlu_full_de', 'global_mmlu_full_id', 'global_mmlu_full_pt', 'global_mmlu_full_ru', 'global_mmlu_full_zh', 'global_mmlu_full_ja', 'global_mmlu_full_ar', 'global_mmlu_full_sw', 'global_mmlu_full_bn', '--batch_size', 'auto', '--output_path', '/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_baseline_olmo2', '--include_path', '/content/lm-evaluation-harness/lm_eval/tasks']' returned non-zero exit status 1.

In [ ]:
# import os

# # Write the eval command to a shell script and run it in a fresh process
# # that has no existing CUDA context
# script = f"""#!/bin/bash
# export VLLM_USE_V1=0
# lm_eval --model vllm \\
#   --model_args "pretrained=allenai/OLMo-2-1124-7B,dtype=auto,gpu_memory_utilization=0.75" \\
#   --tasks global_mmlu_full_en global_mmlu_full_es global_mmlu_full_fr \\
#           global_mmlu_full_de global_mmlu_full_id global_mmlu_full_pt \\
#           global_mmlu_full_ru global_mmlu_full_zh global_mmlu_full_ja \\
#           global_mmlu_full_ar global_mmlu_full_sw global_mmlu_full_bn \\
#   --batch_size auto \\
#   --output_path {RESULTS_BASE} \\
#   --include_path /content/lm-evaluation-harness/lm_eval/tasks
# """

# with open("/content/run_eval.sh", "w") as f:
#     f.write(script)

# os.system("chmod +x /content/run_eval.sh")

# # Run in subprocess with its own clean CUDA context
# # The & at the end runs it in background so the cell doesn't block
# os.system("bash /content/run_eval.sh > /content/eval_baseline_log.txt 2>&1")

# # Monitor progress
# os.system("tail -f /content/eval_baseline_log.txt")

In [1]:
import os
import json
import glob

# Replace the placeholder zeros with your actual baseline dictionary values (as decimals)
base_scores = {
    "global_mmlu_full_en": 0.5649,
    "global_mmlu_full_es": 0.4070,
    "global_mmlu_full_fr": 0.4175,
    "global_mmlu_full_de": 0.4246,
    "global_mmlu_full_id": 0.3719,
    "global_mmlu_full_pt": 0.3684,
    "global_mmlu_full_ru": 0.3158,
    "global_mmlu_full_zh": 0.3193,
    "global_mmlu_full_ja": 0.3298,
    "global_mmlu_full_ar": 0.3368,
    "global_mmlu_full_sw": 0.2982,
    "global_mmlu_full_bn": 0.2526,
}

RESULTS_FT = "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_finetuned_olmo2"

TASKS = [
    "global_mmlu_full_en", "global_mmlu_full_es", "global_mmlu_full_fr",
    "global_mmlu_full_de", "global_mmlu_full_id", "global_mmlu_full_pt",
    "global_mmlu_full_ru", "global_mmlu_full_zh", "global_mmlu_full_ja",
    "global_mmlu_full_ar", "global_mmlu_full_sw", "global_mmlu_full_bn",
]

print("\n" + "="*65)
print(f"{'Language':<15} {'Baseline':>10} {'Finetuned':>10} {'Delta':>10}")
print("="*65)

def load_scores(results_dir):
    scores = {}
    for fpath in glob.glob(os.path.join(results_dir, "**", "*.json"), recursive=True):
        try:
            with open(fpath, "r") as f:
                data = json.load(f)
            results = data.get("results", {})
            for task, metrics in results.items():
                acc = metrics.get("acc,none") or metrics.get("acc_norm,none")
                if acc is None:
                    continue
                parent = next((t for t in TASKS if task.startswith(t)), None)
                if parent:
                    scores.setdefault(parent, []).append(acc)
        except Exception:
            continue
    return {t: sum(v)/len(v) for t, v in scores.items()}

LANG_LABELS = {
    "global_mmlu_full_en": "English",   "global_mmlu_full_es": "Spanish",
    "global_mmlu_full_fr": "French",    "global_mmlu_full_de": "German",
    "global_mmlu_full_id": "Indonesian","global_mmlu_full_pt": "Portuguese",
    "global_mmlu_full_ru": "Russian",   "global_mmlu_full_zh": "Chinese",
    "global_mmlu_full_ja": "Japanese",  "global_mmlu_full_ar": "Arabic",
    "global_mmlu_full_sw": "Swahili",   "global_mmlu_full_bn": "Bengali",
}

# Load only the finetuned scores from disk
ft_scores = load_scores(RESULTS_FT)

for task in TASKS:
    label = LANG_LABELS.get(task, task)
    b     = base_scores.get(task)
    f     = ft_scores.get(task)
    delta = (f - b) if (b and f) else None

    b_str = f"{b*100:.2f}%" if b is not None else "N/A"
    f_str = f"{f*100:.2f}%" if f is not None else "N/A"
    d_str = f"{delta*100:+.2f}pp" if delta is not None else "N/A"

    print(f"{label:<15} {b_str:>10} {f_str:>10} {d_str:>10}")

both = [(base_scores[t], ft_scores[t]) for t in TASKS if t in base_scores and t in ft_scores]
if both:
    avg_b = sum(b for b, _ in both) / len(both)
    avg_f = sum(f for _, f in both) / len(both)
    print("="*65)
    print(f"{'AVERAGE':<15} {avg_b*100:>9.2f}% {avg_f*100:>9.2f}% {(avg_f-avg_b)*100:>+9.2f}pp")


Language          Baseline  Finetuned      Delta
English             56.49%     59.29%    +2.80pp
Spanish             40.70%     43.60%    +2.90pp
French              41.75%     45.44%    +3.69pp
German              42.46%     45.43%    +2.97pp
Indonesian          37.19%     40.90%    +3.71pp
Portuguese          36.84%     45.94%    +9.10pp
Russian             31.58%     39.09%    +7.51pp
Chinese             31.93%     41.81%    +9.88pp
Japanese            32.98%     38.48%    +5.50pp
Arabic              33.68%     36.00%    +2.32pp
Swahili             29.82%     35.47%    +5.65pp
Bengali             25.26%     34.46%    +9.20pp
AVERAGE             36.72%     42.16%     +5.44pp


In [ ]:
import os
import json
import glob

# Ensure variables are defined in this cell
RESULTS_BASE = "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_baseline_olmo2"
RESULTS_FT   = "/content/drive/MyDrive/UCL/SNLP/eval_results/mmlu_dev_finetuned_olmo2"

TASKS = [
    "global_mmlu_full_en", "global_mmlu_full_es", "global_mmlu_full_fr",
    "global_mmlu_full_de", "global_mmlu_full_id", "global_mmlu_full_pt",
    "global_mmlu_full_ru", "global_mmlu_full_zh", "global_mmlu_full_ja",
    "global_mmlu_full_ar", "global_mmlu_full_sw", "global_mmlu_full_bn",
]

print("\n" + "="*65)
print(f"{'Language':<15} {'Baseline':>10} {'Finetuned':>10} {'Delta':>10}")
print("="*65)

def load_scores(results_dir):
    scores = {}
    for fpath in glob.glob(os.path.join(results_dir, "**", "*.json"), recursive=True):
        try:
            with open(fpath, "r") as f:
                data = json.load(f)
            results = data.get("results", {})
            for task, metrics in results.items():
                acc = metrics.get("acc,none") or metrics.get("acc_norm,none")
                if acc is None:
                    continue
                parent = next((t for t in TASKS if task.startswith(t)), None)
                if parent:
                    scores.setdefault(parent, []).append(acc)
        except Exception:
            continue
    return {t: sum(v)/len(v) for t, v in scores.items()}

LANG_LABELS = {
    "global_mmlu_full_en": "English",   "global_mmlu_full_es": "Spanish",
    "global_mmlu_full_fr": "French",    "global_mmlu_full_de": "German",
    "global_mmlu_full_id": "Indonesian","global_mmlu_full_pt": "Portuguese",
    "global_mmlu_full_ru": "Russian",   "global_mmlu_full_zh": "Chinese",
    "global_mmlu_full_ja": "Japanese",  "global_mmlu_full_ar": "Arabic",
    "global_mmlu_full_sw": "Swahili",   "global_mmlu_full_bn": "Bengali",
}

base_scores = load_scores(RESULTS_BASE)
ft_scores   = load_scores(RESULTS_FT)

for task in TASKS:
    label = LANG_LABELS.get(task, task)
    b     = base_scores.get(task)
    f     = ft_scores.get(task)
    delta = (f - b) if (b and f) else None

    b_str = f"{b*100:.2f}%" if b else "N/A"
    f_str = f"{f*100:.2f}%" if f else "N/A"
    d_str = f"{delta*100:+.2f}pp" if delta else "N/A"

    print(f"{label:<15} {b_str:>10} {f_str:>10} {d_str:>10}")

both = [(base_scores[t], ft_scores[t]) for t in TASKS if t in base_scores and t in ft_scores]
if both:
    avg_b = sum(b for b, _ in both) / len(both)
    avg_f = sum(f for _, f in both) / len(both)
    print("="*65)
    print(f"{'AVERAGE':<15} {avg_b*100:>9.2f}% {avg_f*100:>9.2f}% {(avg_f-avg_b)*100:>+9.2f}pp")

print("\nAll results saved to Drive.")